# 05 — Valorización esperada futura 2026–2031

**Sección 6 del Informe 1 — Grupo 13 FinPUC**

Este notebook produce:
- **6.1** Proyección determinística: $V_t = V_0(1+\mu_p)^t$
- **6.2** Proyección bootstrap con bandas de incertidumbre P5 / P50 / P95
- **Tabla 5**: Resumen 2031 para los 5 perfiles y el benchmark

**Dependencias:** ejecutar primero `01_Preparar_Datos.ipynb`, `02_Modelo_Markowitz.ipynb` y `03_Perfiles_Riesgo.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

OUTPUTS = Path('outputs')
FIG_DIR  = OUTPUTS / 'figuras'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Parámetros de proyección ───────────────────────────────────────────────
V0       = 1_000          # capital inicial USD
HORIZONTE_ANIOS = 5       # 2026–2031
SEMANAS  = 52 * HORIZONTE_ANIOS   # 260 semanas
N_SIM    = 10_000          # trayectorias bootstrap
RNG      = np.random.default_rng(42)

# Fecha de inicio: primera semana de enero 2026
inicio   = pd.Timestamp('2026-01-05')
fechas   = pd.date_range(inicio, periods=SEMANAS, freq='W-MON')

print(f'Horizonte: {inicio.date()} → {fechas[-1].date()} ({SEMANAS} semanas)')

In [ ]:
# ── Cargar métricas pre-computadas de los perfiles ────────────────────────
summary = pd.read_csv(OUTPUTS / 'markowitz_profiles_summary.csv')
print(summary.to_string(index=False))

In [ ]:
# ── Cargar retornos semanales (para bootstrap) ────────────────────────────
weekly_returns = pd.read_pickle(OUTPUTS / 'weekly_returns.pkl')   # shape: (semanas, n_acciones)
pesos          = pd.read_csv(OUTPUTS / 'markowitz_profile_weights.csv', index_col=0)

# Retorno semanal del portafolio para cada perfil
# Si los pesos están en columnas = perfiles, filas = tickers
perfiles_ids = pesos.columns.tolist()
print('Perfiles cargados:', perfiles_ids)
print(f'Retornos semanales: {weekly_returns.shape}')

## 6.1 Proyección determinística

$$V_t = V_0 \left(1 + \mu_p\right)^t$$

donde $\mu_p$ es el retorno **semanal** esperado del portafolio:
$$\mu_p^{\text{sem}} = (1 + \mu_p^{\text{anual}})^{1/52} - 1$$

In [ ]:
def retorno_semanal(mu_anual: float) -> float:
    return (1 + mu_anual) ** (1 / 52) - 1


def trayectoria_deterministica(mu_anual: float, v0: float = V0, semanas: int = SEMANAS) -> np.ndarray:
    mu_sem = retorno_semanal(mu_anual)
    t      = np.arange(1, semanas + 1)
    return v0 * (1 + mu_sem) ** t


# Construir trayectorias determinísticas para cada perfil
det_paths = {}
for _, row in summary.iterrows():
    perfil    = row.get('perfil', row.get('profile', str(_)))
    mu_anual  = row.get('retorno_anual', row.get('expected_return', 0))
    det_paths[perfil] = trayectoria_deterministica(mu_anual)

print('Trayectorias determinísticas construidas:', list(det_paths.keys()))

In [ ]:
# ── Figura 2: Valorización esperada ───────────────────────────────────────
COLORES = {
    'muy_conservador' : '#2196F3',
    'conservador'     : '#4CAF50',
    'neutro'          : '#FF9800',
    'arriesgado'      : '#F44336',
    'muy_arriesgado'  : '#9C27B0',
    'benchmark'       : '#607D8B',
}

fig, ax = plt.subplots(figsize=(12, 6))

for perfil, trayectoria in det_paths.items():
    color = COLORES.get(perfil, '#888')
    ls    = '--' if 'benchmark' in str(perfil).lower() else '-'
    ax.plot(fechas, trayectoria, label=perfil, color=color, linewidth=2, linestyle=ls)

ax.axhline(V0, color='gray', linewidth=0.8, linestyle=':', label='Capital inicial')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xlabel('Año')
ax.set_ylabel('Capital (USD)')
ax.set_title('Figura 2: Valorización esperada de portafolios 2026–2031')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'figura2_valorizacion_deterministica.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura 2 guardada.')

## 6.2 Proyección bootstrap con bandas de incertidumbre

Se re-muestrean retornos semanales históricos del portafolio con reemplazo para construir $N=10\,000$ trayectorias.
Para cada fecha $t$ se calculan los percentiles P5, P50 y P95.

In [ ]:
def bootstrap_paths(
    retornos_semanales: np.ndarray,   # shape (T_hist,)
    v0: float = V0,
    semanas: int = SEMANAS,
    n_sim: int = N_SIM,
    rng=None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Retorna (p5, p50, p95) — arrays de shape (semanas,).
    """
    if rng is None:
        rng = np.random.default_rng(42)

    # Índices con reemplazo para re-muestrar
    idx   = rng.integers(0, len(retornos_semanales), size=(n_sim, semanas))
    draws = retornos_semanales[idx]           # shape (n_sim, semanas)

    # Acumular retorno compuesto
    paths = v0 * np.cumprod(1 + draws, axis=1)   # (n_sim, semanas)

    p5  = np.percentile(paths, 5,  axis=0)
    p50 = np.percentile(paths, 50, axis=0)
    p95 = np.percentile(paths, 95, axis=0)
    return p5, p50, p95


# Calcular retornos semanales del portafolio para cada perfil
boot_results = {}
for perfil in pesos.columns:
    w = pesos[perfil].values                      # pesos del perfil
    tickers_disponibles = [t for t in pesos.index if t in weekly_returns.columns]
    w_validos = pesos.loc[tickers_disponibles, perfil].values
    w_validos = w_validos / w_validos.sum()        # renormalizar

    ret_portafolio = (weekly_returns[tickers_disponibles] * w_validos).sum(axis=1).values
    p5, p50, p95   = bootstrap_paths(ret_portafolio)
    boot_results[perfil] = {'p5': p5, 'p50': p50, 'p95': p95}
    print(f'{perfil}: P5_2031={p5[-1]:,.0f}  P50={p50[-1]:,.0f}  P95={p95[-1]:,.0f}')

In [ ]:
# ── Figura 3: Bandas bootstrap ────────────────────────────────────────────
n_perfiles = len(boot_results)
fig, axes  = plt.subplots(2, 3, figsize=(16, 8), sharey=False)
axes_flat  = axes.flatten()

for idx_ax, (perfil, bands) in enumerate(boot_results.items()):
    if idx_ax >= len(axes_flat):
        break
    ax     = axes_flat[idx_ax]
    color  = COLORES.get(perfil, '#888')
    det    = det_paths.get(perfil, bands['p50'])

    ax.fill_between(fechas, bands['p5'], bands['p95'], alpha=0.2, color=color, label='Banda P5–P95')
    ax.plot(fechas, bands['p95'], color=color, linewidth=0.8, linestyle=':')
    ax.plot(fechas, bands['p50'], color=color, linewidth=2,   label='P50 (mediana)')
    ax.plot(fechas, bands['p5'],  color=color, linewidth=0.8, linestyle=':')
    ax.plot(fechas, det,          color='gray', linewidth=1.2, linestyle='--', label='Determinística')
    ax.axhline(V0, color='black', linewidth=0.6, linestyle=':')

    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.set_title(perfil.replace('_', ' ').title(), fontsize=10)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

# Ocultar ejes sobrantes
for ax in axes_flat[n_perfiles:]:
    ax.set_visible(False)

fig.suptitle('Figura 3: Proyección con bandas bootstrap P5–P95 (2026–2031)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'figura3_bootstrap_bandas.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura 3 guardada.')

In [ ]:
# ── Tabla 5: Resumen 2026–2031 ────────────────────────────────────────────
filas = []
for _, row in summary.iterrows():
    perfil   = row.get('perfil', row.get('profile', str(_)))
    mu_anual = row.get('retorno_anual', row.get('expected_return', 0))
    vol_anual = row.get('volatilidad_anual', row.get('volatility', 0))
    det_2031 = det_paths.get(perfil, np.array([np.nan]))[-1]

    if perfil in boot_results:
        p5_2031  = boot_results[perfil]['p5'][-1]
        p50_2031 = boot_results[perfil]['p50'][-1]
        p95_2031 = boot_results[perfil]['p95'][-1]
    else:
        p5_2031 = p50_2031 = p95_2031 = np.nan

    filas.append({
        'Portafolio'   : perfil.replace('_', ' ').title(),
        'Ret. anual'   : f"{mu_anual*100:.1f} %",
        'Vol. anual'   : f"{vol_anual*100:.1f} %",
        'Esperado 2031': f"${det_2031:,.0f}",
        'P5 2031'      : f"${p5_2031:,.0f}",
        'P50 2031'     : f"${p50_2031:,.0f}",
        'P95 2031'     : f"${p95_2031:,.0f}",
    })

tabla5 = pd.DataFrame(filas)
print('\nTabla 5 — Resumen de proyección futura 2026–2031\n')
print(tabla5.to_string(index=False))
tabla5.to_csv(OUTPUTS / 'tabla5_proyeccion_2031.csv', index=False)
print('\nGuardada en outputs/tabla5_proyeccion_2031.csv')

## Interpretación

- La **proyección determinística** (Ecuación 13) muestra la trayectoria esperada de riqueza asumiendo retorno constante.
- La **proyección bootstrap** revela que la amplitud P5–P95 crece con el tiempo y con el nivel de riesgo del perfil.
- Desde una perspectiva ajustada por riesgo, el **perfil neutro** obtiene mayor riqueza esperada que el benchmark con menor volatilidad histórica.
- El **benchmark equiponderado** se ubica entre el perfil conservador y neutro en valor mediano, pero con volatilidad cercana al perfil arriesgado, lo que lo hace ineficiente en la frontera riesgo-retorno.

Referencia: Secciones 6.1 y 6.2, Tabla 5 y Figuras 2–3 del Informe 1 — Grupo 13.